In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# 1. 수집 설정 (야후 파이낸스용 삼성전자 티커: 005930.KS)
ticker = "005930.KS"
start_date = "2006-06-01"
end_date = "2026-05-31"

print(f"yfinance를 통해 데이터를 수집합니다. (기간: {start_date} ~ {end_date})")
df = yf.download(ticker, start=start_date, end=end_date)

# 2. yfinance 데이터 정리 (MultiIndex 컬럼인 경우 단일 레벨로 평탄화)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# 요청하신 필드명으로 컬럼명 변경
df = df.rename(columns={
    'Open': 'stck_oprc',
    'High': 'stck_hgpr',
    'Low': 'stck_lwpr',
    'Close': 'stck_clpr',
    'Volume': 'acml_vol'
})

# 결측치(휴장일 등) 제거
df = df.dropna()

# 3. 추가 파생 변수 계산
# 전일대비 (당일 종가 - 전일 종가)
df['prdy_vrss'] = df['stck_clpr'].diff()

# 전일대비부호 (양수 1, 음수 -1, 보합 0)
df['prdy_vrss_sign'] = np.sign(df['prdy_vrss'])

# 전일대비거래량증감율 (%)
df['prdy_vrss_vol_rate'] = df['acml_vol'].pct_change() * 100

# 전일대비율 (등락률, %)
df['prdy_ctrt'] = df['stck_clpr'].pct_change() * 100

# 누적수익률 (첫날 종가 기준 누적)
df['acml_prtt_rate'] = (1 + df['prdy_ctrt'] / 100).cumprod() - 1

# 계산 중 발생한 첫날 결측치(NaN) 0으로 처리
df = df.fillna(0)

# 인덱스 이름 설정
df.index.name = 'stck_bsop_date'

# 4. 필요한 컬럼만 추출 (외국인 데이터는 제외)
final_columns = [
    'stck_oprc', 'stck_hgpr', 'stck_lwpr', 'stck_clpr', 'acml_vol',
    'prdy_vrss', 'prdy_ctrt', 'prdy_vrss_sign', 'prdy_vrss_vol_rate',
    'acml_prtt_rate'
]
df_final = df[final_columns]

# 5. CSV 파일로 저장
file_name = "Samsung_Daily_Data_yfinance.csv"
df_final.to_csv(file_name)
print(f"Successfully generated {file_name}")

yfinance를 통해 데이터를 수집합니다. (기간: 2006-06-01 ~ 2026-05-31)


/tmp/ipykernel_12275/1836761480.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Successfully generated Samsung_Daily_Data_yfinance.csv
